# Cox Integration

In [1]:
import time
import pickle
import json
from lifelines import CoxPHFitter
import copy
import os
import torch

## model auditing
from pathlib import Path
import numpy as np
import torch.utils.data
from sklearn.metrics import roc_curve, auc
from torch.utils.data import Subset
from attacks import tune_offline_a, run_rmia, run_loss
from visualize import plot_roc, plot_roc_log

## build model and dataset
import pandas as pd
from lifelines.datasets import load_rossi
import matplotlib.pyplot as plt

## synthetic dataset
import glob

In [53]:
global_log_dir = "cox_synthetic_dataset/signal_cindex_small_dataset_1000"

## split dataset for training

In [8]:
# whole dataset
def split_dataframe_for_training(dataframe, num_model_pairs):
    """
    Split a Pandas DataFrame into training and test partitions for model pairs.

    Args:
        dataframe (pd.DataFrame): Input dataset as a Pandas DataFrame.
        num_model_pairs (int): Number of model pairs to be trained, with each pair trained on different halves of the dataset.

    Returns:
        data_splits (list): List of dictionaries containing training and test DataFrames for each model.
        master_keep (np.array): Boolean array indicating the membership of samples in each model's training set.
    """
    dataset_size = len(dataframe)
    indices = np.arange(dataset_size)
    split_index = dataset_size // 2
    master_keep = np.full((2 * num_model_pairs, dataset_size), True, dtype=bool)
    data_splits = []

    for i in range(num_model_pairs):
        # Shuffle indices to randomize the dataset
        np.random.shuffle(indices)
        
        # Update master_keep for training and testing sets
        master_keep[i * 2, indices[split_index:]] = False
        master_keep[i * 2 + 1, indices[:split_index]] = False
        
        # Generate train and test indices
        train_indices_1 = np.where(master_keep[i * 2, :])[0]
        test_indices_1 = np.where(~master_keep[i * 2, :])[0]

        train_indices_2 = np.where(master_keep[i * 2 + 1, :])[0]
        test_indices_2 = np.where(~master_keep[i * 2 + 1, :])[0]
        
        # Append training and testing DataFrames for both models in the pair
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_1],
                "test": dataframe.iloc[test_indices_1],
            }
        )
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_2],
                "test": dataframe.iloc[test_indices_2],
            }
        )

    return data_splits, master_keep

In [4]:
# non-censored dataset
def split_dataframe_for_training(dataframe, num_model_pairs):
    """
    Split a Pandas DataFrame into training and test partitions, ensuring each 
    model's training set contains exactly half of the non-censored records 
    (where 'arrest' == 1).

    Args:
        dataframe (pd.DataFrame): Input dataset as a Pandas DataFrame.
        num_model_pairs (int): Number of model pairs to be trained.

    Returns:
        data_splits (list): List of dictionaries containing training and 
                           test DataFrames for each model.
        master_keep (np.array): Boolean array indicating sample membership.
    """

    # Identify non-censored records (arrest == 1)
    non_censored_indices = dataframe[dataframe['arrest'] == 1].index.to_numpy()
    num_non_censored = len(non_censored_indices)

    mp = {}
    val = 0
    for idx in non_censored_indices:
        mp[idx] = val
        val += 1
    
    # Handle censored records (arrest == 0):
    censored_indices = dataframe[dataframe['arrest'] == 0].index.to_numpy()
    num_censored = len(censored_indices)

    dataset_size = len(dataframe)
    indices = np.arange(dataset_size)
    master_keep = np.full((2 * num_model_pairs, len(non_censored_indices)), False, dtype=bool)
    data_splits = []

    for i in range(num_model_pairs):
        np.random.shuffle(censored_indices)
        np.random.shuffle(non_censored_indices)
        
        # non censored
        split_index_non_censored = num_non_censored // 2

        train_non_censored_indices_1 = non_censored_indices[:split_index_non_censored]
        test_non_censored_indices_1 = non_censored_indices[split_index_non_censored:]

        train_non_censored_indices_2 = non_censored_indices[split_index_non_censored:]
        test_non_censored_indices_2 = non_censored_indices[:split_index_non_censored]
        
        # censored
        split_index_censored = len(censored_indices) // 2

        train_censored_indices_1 = censored_indices[:split_index_censored]
        test_censored_indices_1 = censored_indices[split_index_censored:]

        train_censored_indices_2 = censored_indices[split_index_censored:]
        test_censored_indices_2 = censored_indices[:split_index_censored]

        # Combine indices for training and testing
        train_indices_1 = np.concatenate([train_non_censored_indices_1, train_censored_indices_1])
        test_indices_1 = np.concatenate([test_non_censored_indices_1, test_censored_indices_1])

        train_indices_2 = np.concatenate([train_non_censored_indices_2, train_censored_indices_2])
        test_indices_2 = np.concatenate([test_non_censored_indices_2, test_censored_indices_2])

        '''
        # Update master_keep (membership)
        master_keep[i * 2, train_indices_1] = True
        master_keep[i * 2, test_indices_1] = False

        master_keep[i * 2 + 1, train_indices_2] = True
        master_keep[i * 2 + 1, test_indices_2] = False
        '''


        for idx in train_non_censored_indices_1:
            master_keep[i * 2, mp[idx]] = True
        for idx in train_non_censored_indices_2:
            master_keep[i * 2 + 1, mp[idx]] = True


        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_1],
                "test": dataframe.iloc[test_indices_1],
            }
        )
        data_splits.append(
            {
                "train": dataframe.iloc[train_indices_2],
                "test": dataframe.iloc[test_indices_2],
            }
        )

    return data_splits, master_keep

## train cox models using the splitted dataset

In [9]:
def train_cox_models(data_splits, num_model_pairs, log_dir):
    """
    Train Cox proportional hazards models using data splits and store metadata.

    Args:
        data_splits (list): List of dictionaries containing "train" and "test" DataFrames.
        num_model_pairs (int): Number of model pairs to train.
        log_dir (str): Directory to store models and metadata.
        logger (logging.Logger): Logger for logging information.
        dataset_name (str): Name of the dataset used for training.

    Returns:
        list: List of trained CoxPHFitter models.
    """
    os.makedirs(log_dir, exist_ok=True)  # Ensure log directory exists
    model_list = []
    model_metadata_dict = {}

    for split_idx, split_info in enumerate(data_splits):
        #print(f"Training model {split_idx}...")

        # Extract train and test sets
        train_data = split_info["train"]
        test_data = split_info["test"]

        # Initialize the CoxPH model
        model = CoxPHFitter()
        baseline_time = time.time()

        # Train the Cox model
        model.fit(train_data, duration_col="Time", event_col="Event")

        # Evaluate the model
        train_score = model.score(train_data, scoring_method="concordance_index")
        test_score = model.score(test_data, scoring_method="concordance_index")

        #train_loss = model.log_likelihood_ratio_
        #test_loss = model.compute_residuals(test_data, "martingale").abs().mean()

       # logger.info(
       #     "Training model %s took %.2f seconds", split_idx, time.time() - baseline_time
       # )

        # Store the trained model
        model_list.append(copy.deepcopy(model))

        model_idx = split_idx

        with open(f"{log_dir}/model_{model_idx}.pkl", "wb") as f:
            pickle.dump(model, f)

        # Store metadata
        model_metadata_dict[model_idx] = {
            "model_name": "CoxPHFitter",
            "num_train": len(train_data),
            "num_test": len(test_data),
            "train_acc": train_score,
            "test_acc": test_score,
            #"train_loss": train_loss,
            #"test_loss": test_loss,
            #"dataset": dataset_name,
            "model_path": f"{log_dir}/cox_model_{model_idx}.pkl",
        }

    # Save all metadata as a JSON file
    with open(f"{log_dir}/cox_models_metadata.json", "w") as f:
        json.dump(model_metadata_dict, f, indent=4)

    return model_list, model_metadata_dict


## prepare auditing dataset and memberships

In [10]:
# use original dataset and memberships (if no downsamples specified)
def sample_cox_auditing_dataset(dataset, memberships):
    return dataset, memberships

In [7]:
# auditing dataset only covers non-censored dataset
def sample_cox_auditing_dataset(dataset: pd.DataFrame, memberships) -> pd.DataFrame:
    """
    Extracts all non-censored records from the dataset where the 'arrest' column is 1.

    Parameters:
    dataset (pd.DataFrame): The input dataset (Rossi dataset).

    Returns:
    pd.DataFrame: A new dataset containing only non-censored records.
    """
    return dataset[dataset['arrest'] == 1].copy(), memberships

## Compute Attack Signals

## compute model signals
- Use "partial likelihood per sample" to generate pseudo "loss" values. use get_loss function to compute signals and do attacks.

- Function: 

def cox_partial_likelihood_per_sample(predicted_risks, true_events, true_times)

- Explaination:

  """
  Calculates the Cox partial likelihood loss for each sample.

  Args:
      predicted_risks (torch.Tensor): Predicted risks from the Cox model.
  
      true_events (torch.Tensor): Indicator vector for events (1 for event, 0 for censored).
  
      true_times (torch.Tensor): Survival times for each sample.

  Returns:
      torch.Tensor: A tensor containing the loss for each sample.
  """

### compute attack signals based on square of distance

In [32]:
# one dataset for one model

def get_cox_model_signals(model_list, data_list, log_dir=global_log_dir):
    signals = []
    model_idx = 0
    
    for model in model_list:
        print(f"Compute attack signals for model {model_idx}...")
        model_idx += 1
        
        predicted_times = torch.tensor(model.predict_expectation(data_list[idx - 1]).values) 
        true_events = torch.tensor(data_list[idx - 1]['Event'].values)
        true_times = torch.tensor(data_list[idx - 1]['Time'].values)

        n = len(predicted_times)
        sample_losses = np.zeros(n)
        
        for i in range(n):
            if true_events[i] == 1:
                if predicted_times[i] == float('inf'):
                    sample_losses[i] = float('inf')
                else:
                    sample_losses[i] = (predicted_times[i] - true_times[i]) * (predicted_times[i] - true_times[i])
            else:
                if predicted_times[i] == float('inf') or predicted_times[i] >= true_times[i]:
                    sample_losses[i] = 0
                else:
                    # option 1: underestimate signals
                    #sample_losses[i] = (predicted_times[i] - true_times[i]) * (predicted_times[i] - true_times[i])
                    
                    # option 2: manually make signals larger
                    correction_value = 100
                    sample_losses[i] = (true_times[i] - predicted_times[i] + correction_value) * (true_times[i] - predicted_times[i] + correction_value)
                    
                    # option 3: cannot deal with infinite, must be postprocessed
                    #sample_losses[i] = 256 #float('inf')
                    
        signals.append(sample_losses.reshape(-1, 1))
        
    signals = np.concatenate(signals, axis = 1)
    np.save(
        f"{log_dir}/cox_square_signals.npy",
        signals,
    )
    print("Signals saved to disk.")
    return signals

### compute loss based on individual c-index

In [71]:
# performance optimization using vectorization

def get_cox_model_signals(model_list, data, log_dir=global_log_dir):
    signals = []
    model_idx = 0

    print(f"Total: {len(data)} signals to be calculated...")
    
    for model in model_list:
        print(f"Compute attack signals for model {model_idx}...")
        model_idx += 1
        
        # Convert data to PyTorch tensors
        predicted_times = torch.tensor(model.predict_expectation(data).values)  
        true_events = torch.tensor(data['Event'].values)
        true_times = torch.tensor(data['Time'].values)
        
        # Get total number of samples
        n = len(predicted_times)
        
        # Expand dimensions to enable broadcasting
        true_times_i = true_times.view(n, 1)
        true_times_j = true_times.view(1, n)
        predicted_times_i = predicted_times.view(n, 1)
        predicted_times_j = predicted_times.view(1, n)
        true_events_i = true_events.view(n, 1)
        true_events_j = true_events.view(1, n)
        
        # Compute masks for different comparison cases
        non_censored_i = true_events_i == 1
        non_censored_j = true_events_j == 1
        censored_i = ~non_censored_i
        
        # Condition 1: Both are non-censored
        valid_pairs_1 = non_censored_i & non_censored_j
        correct_pairs_1 = ((true_times_i > true_times_j) == (predicted_times_i > predicted_times_j)).float()
        correct_pairs_1 += ((true_times_i == true_times_j) & (predicted_times_i == predicted_times_j)).float() * 0.5
        
        # Condition 2: i is non-censored, j is censored & j's time is >= i's time
        valid_pairs_2 = non_censored_i & ~non_censored_j & (true_times_j >= true_times_i)
        correct_pairs_2 = (predicted_times_j > predicted_times_i).float()
        
        # Condition 3: i is censored, j is non-censored & i's time is >= j's time
        valid_pairs_3 = censored_i & non_censored_j & (true_times_i >= true_times_j)
        correct_pairs_3 = (predicted_times_i > predicted_times_j).float()
        
        # Compute total valid pairs for each `i`
        total_pairs = valid_pairs_1.float().sum(dim=1) + valid_pairs_2.float().sum(dim=1) + valid_pairs_3.float().sum(dim=1)
        
        # Compute total correct pairs for each `i`
        correct_pairs = (valid_pairs_1 * correct_pairs_1).sum(dim=1) + (valid_pairs_2 * correct_pairs_2).sum(dim=1) + (valid_pairs_3 * correct_pairs_3).sum(dim=1)
        
        # Compute sample losses
        sample_losses = torch.where(total_pairs > 0, correct_pairs / total_pairs, torch.zeros_like(total_pairs))

    
        signals.append(sample_losses.reshape(-1, 1))
        
    signals = np.concatenate(signals, axis = 1)
    np.save(
        f"{log_dir}/cox_cindex_signals.npy",
        signals,
    )
    print("Signals saved to disk.")
    return signals
            

## audit model using attack signals (pseudo "loss" values)

In [14]:
def run_cox_loss(target_signals: np.ndarray) -> np.ndarray:
    """
    Attack a target model using the LOSS attack.

    Args:
        target_signals (np.ndarray): Softmax value of all samples in the target model.

    Returns:
        np.ndarray: MIA score for all samples (a larger score indicates higher chance of being member).
    """
    mia_scores = -target_signals  # for square signals
    #mia_scores = target_signals  # for c-index signals
    return mia_scores

def compute_cox_attack_results(mia_scores, target_memberships):
    """
    Compute attack results (TPR-FPR curve, AUC, etc.) based on MIA scores and membership of samples.

    Args:
        mia_scores (np.array): MIA score computed by the attack.
        target_memberships (np.array): Membership of samples in the training set of target model.

    Returns:
        dict: Dictionary of results, including fpr and tpr list, AUC, TPR at 1%, 0.1% and 0% FPR.
    """
    fpr_list, tpr_list, _ = roc_curve(target_memberships.ravel(), mia_scores.ravel())
    roc_auc = auc(fpr_list, tpr_list)
    one_fpr = tpr_list[np.where(fpr_list <= 0.01)[0][-1]]
    one_tenth_fpr = tpr_list[np.where(fpr_list <= 0.001)[0][-1]]
    zero_fpr = tpr_list[np.where(fpr_list <= 0.0)[0][-1]]

    return {
        "fpr": fpr_list,
        "tpr": tpr_list,
        "auc": roc_auc,
        "one_fpr": one_fpr,
        "one_tenth_fpr": one_tenth_fpr,
        "zero_fpr": zero_fpr,
    }

def get_cox_audit_results(report_dir, model_idx, mia_scores, target_memberships):
    """
    Generate and save ROC plots for attacking a single model.

    Args:
        report_dir (str): Folder for saving the ROC plots.
        model_idx (int): Index of model subjected to the attack.
        mia_scores (np.array): MIA score computed by the attack.
        target_memberships (np.array): Membership of samples in the training set of target model.
        logger (logging.Logger): Logger object for the current run.

    Returns:
        dict: Dictionary of results, including fpr and tpr list, AUC, TPR at 1%, 0.1% and 0% FPR.
    """
    attack_result = compute_cox_attack_results(mia_scores, target_memberships)
    Path(report_dir).mkdir(parents=True, exist_ok=True)

    print(
        f"Target Model {model_idx}: AUC {attack_result['auc']:.4f}, "
        #f"TPR@0.1%FPR {attack_result['one_tenth_fpr']:.4f}, "
        #f"TPR@0.0%FPR {attack_result['zero_fpr']:.4f}"
    )

    plot_roc(
        attack_result["fpr"],
        attack_result["tpr"],
        attack_result["auc"],
        f"{report_dir}/ROC_{model_idx}.png",
    )
    plot_roc_log(
        attack_result["fpr"],
        attack_result["tpr"],
        attack_result["auc"],
        f"{report_dir}/ROC_log_{model_idx}.png",
    )

    np.savez(
        f"{report_dir}/attack_result_{model_idx}",
        fpr=attack_result["fpr"],
        tpr=attack_result["tpr"],
        auc=attack_result["auc"],
        one_tenth_fpr=attack_result["one_tenth_fpr"],
        zero_fpr=attack_result["zero_fpr"],
        scores=mia_scores.ravel(),
        memberships=target_memberships.ravel(),
    )
    return attack_result

def audit_cox_models(
    target_model_indices,
    all_signals,
    all_memberships,
    report_dir=global_log_dir
):
    """
    Audit target model(s) using a Membership Inference Attack algorithm.

    Args:
        report_dir (str): Folder to save attack result.
        target_model_indices (list): List of the target model indices.
        all_signals (np.array): Signal value of all samples in all models (target and reference models).
        all_memberships (np.array): Membership matrix for all models.
        num_reference_models (int): Number of reference models used for performing the attack.
        logger (logging.Logger): Logger object for the current run.
        configs (dict): Configs provided by the user.

    Returns:
        list: List of MIA score arrays for all audited target models.
        list: List of membership labels for all target models.
    """
    all_memberships = np.transpose(all_memberships)

    mia_score_list = []
    membership_list = []

    for target_model_idx in target_model_indices:
        print(
            f"Auditing the privacy risks of target model {target_model_idx}"
        )
        
        mia_scores = run_cox_loss(all_signals[:, target_model_idx])
        target_memberships = all_memberships[:, target_model_idx]

        mia_score_list.append(mia_scores.copy())
        membership_list.append(target_memberships.copy())

        _ = get_cox_audit_results(
            report_dir, target_model_idx, mia_scores, target_memberships
        )

    return mia_score_list, membership_list

# Synthetic Dataset

## First synthetic dataset - size 20000 (2011 censored, 17989 non-censored)

In [74]:
# read the first synthetic dataset - size 20000
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset/Reference_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
data.drop('dataset', axis=1, inplace=True)

print(data.head())

global_log_dir = "cox_synthetic_dataset/signal_cindex_dataset_size20000"

         Age      Time  Event  Sex_F  Treatment_B  Treatment_C
0  35.640666  3.354256    1.0    1.0          0.0          0.0
1  43.846724  7.610293    1.0    1.0          1.0          0.0
2  54.546209  0.769248    1.0    1.0          0.0          1.0
3  26.964070  3.894167    1.0    1.0          0.0          1.0
4  23.970407  0.380061    1.0    1.0          0.0          1.0


In [ ]:
count_zero_events = (data['Event'] == 0).sum()
print(f"Number of rows with 'Event' equal to zero: {count_zero_events}")
count_one_events = (data['Event'] == 1).sum()
print(f"Number of rows with 'Event' equal to one: {count_one_events}")

## Build Cox model

In [15]:
# Initialize the Cox Proportional Hazards model
cox_model = CoxPHFitter()
# Fit the model to the data
cox_model.fit(data, duration_col='Time', event_col='Event')
cox_model.print_summary()

<lifelines.CoxPHFitter: fitted with 3000 total observations, 292 right-censored observations>
             duration col = 'Time'
                event col = 'Event'
      baseline estimation = breslow
   number of observations = 3000
number of events observed = 2708
   partial log-likelihood = -18835.47
         time fit was run = 2025-02-19 20:58:57 UTC

---
             coef exp(coef)  se(coef)  coef lower 95%  coef upper 95% exp(coef) lower 95% exp(coef) upper 95%
covariate                                                                                                    
Age          0.01      1.01      0.00            0.01            0.01                1.01                1.01
Sex_F        0.64      1.91      0.07            0.51            0.78                1.67                2.17
Treatment_B -0.24      0.79      0.05           -0.33           -0.15                0.72                0.86
Treatment_C  0.26      1.30      0.05            0.17            0.36                1.19                1.43

             cmp to     z      p  -log2(p)
covariate                                 
Age            0.00  7.89 <0.005     48.28
Sex_F          0.00  9.56 <0.005     69.55
Treatment_B    0.00 -5.05 <0.005     21.08
Treatment_C    0.00  5.63 <0.005     25.69
---
Concordance = 0.59
Partial AIC = 37678.94
log-likelihood ratio test = 261.65 on 4 df
-log2(p) of ll-ratio test = 181.70

In [ ]:
## optional

# Make predictions - We can predict the survival function for each individual
# Here, we predict the survival function for the first 5 individuals
print("\nPredicted survival functions for the first 5 individuals:")
survival_functions = cox_model.predict_survival_function(data.iloc[:5])
print(survival_functions)

# Plot the survival functions
plt.figure(figsize=(10, 6))
for i in range(survival_functions.shape[1]):
    plt.step(survival_functions.index, survival_functions.iloc[:, i], where="post", label=f"Individual {i+1}")
plt.title("Predicted Survival Functions")
plt.xlabel("Time (weeks)")
plt.ylabel("Survival Probability")
plt.legend()
plt.show()

### workflow

In [100]:
## Split dataset randomly

num_model_pairs = 10
data_splits, memberships = split_dataframe_for_training(data, num_model_pairs)

In [101]:
## Train cox models and get trained models and metadata

models, metadata = train_cox_models(
    data_splits=data_splits,
    num_model_pairs=num_model_pairs,
    log_dir=global_log_dir
)

#print(metadata[0])  # Metadata for the first model
#models[0].print_summary()  # Summary of the first model

In [102]:
## prepare auditing dataset and memberships

auditing_dataset, auditing_membership = sample_cox_auditing_dataset(data, memberships)

In [103]:
## compute attack signals
signals = get_cox_model_signals(models, auditing_dataset, global_log_dir)

Compute attack signals for model 0...
Compute attack signals for model 1...
Compute attack signals for model 2...
Compute attack signals for model 3...
Compute attack signals for model 4...
Compute attack signals for model 5...
Compute attack signals for model 6...
Compute attack signals for model 7...
Compute attack signals for model 8...
Compute attack signals for model 9...
Compute attack signals for model 10...
Compute attack signals for model 11...
Compute attack signals for model 12...
Compute attack signals for model 13...
Compute attack signals for model 14...
Compute attack signals for model 15...
Compute attack signals for model 16...
Compute attack signals for model 17...
Compute attack signals for model 18...
Compute attack signals for model 19...
Signals saved to disk.


### Attack signals = Square, all data, version 1

In [91]:
global_log_dir = "cox_synthetic_dataset/signal_square_dataset_20000_version1"

In [97]:
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.5036, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.4964, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.4979, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.5019, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.5060, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.4938, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.4986, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.5013, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.5027, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.4971, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.5018, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.4979, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.4987, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

### Attack signals = Square, all data, version 2

In [99]:
global_log_dir = "cox_synthetic_dataset/signal_square_dataset_20000_version2"

In [105]:
# Perform the privacy auditing
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.4939, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.5063, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.4988, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.5016, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.5008, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.4995, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.4994, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.5021, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.4956, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.5042, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.4931, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.5074, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.4954, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

### Attack signals = individual c-index

In [86]:
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.4964, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.5036, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.5021, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.4981, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.4940, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.5062, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.5014, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.4987, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.4973, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.5029, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.4982, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.5021, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.5013, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

### Attack signals = C-index + normalization

In [87]:
# Normalization layer of the attack signals
def run_cox_loss(target_signals: np.ndarray) -> np.ndarray:
    # Avoid division by zero if all values are the same
    min_val = np.min(target_signals)
    max_val = np.max(target_signals)
    
    if max_val == min_val:
        return np.zeros_like(target_signals)  # If all values are the same, return zeros
    
    # Normalize to [0, 1]
    mia_scores = (target_signals - min_val) / (max_val - min_val)
    
    return mia_scores

In [88]:
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.4964, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.5036, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.5021, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.4981, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.4940, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.5062, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.5014, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.4987, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.4973, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.5029, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.4982, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.5021, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.5013, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

## Smaller dataset - size 400

In [42]:
data_small = data.sample(n=400, random_state=1)  # 400 rows
print(len(data_small))  # Output: 400
print(data_small.head())

400
             Age      Time  Event  Sex_F  Treatment_B  Treatment_C
11456  28.890721  3.449446    0.0    1.0          0.0          1.0
16528  19.526446  8.390862    0.0    1.0          0.0          1.0
3253   40.257764  6.339574    1.0    0.0          0.0          0.0
18614  21.156177  6.108753    1.0    1.0          0.0          0.0
1544   35.465065  0.606655    1.0    1.0          0.0          0.0


## Smaller dataset - size 1000

In [55]:
data_small = data.sample(n=1000, random_state=1)  # 1000 rows
print(len(data_small))  # Output: 1000
print(data_small.head())

1000
             Age      Time  Event  Sex_F  Treatment_B  Treatment_C
11456  28.890721  3.449446    0.0    1.0          0.0          1.0
16528  19.526446  8.390862    0.0    1.0          0.0          1.0
3253   40.257764  6.339574    1.0    0.0          0.0          0.0
18614  21.156177  6.108753    1.0    1.0          0.0          0.0
1544   35.465065  0.606655    1.0    1.0          0.0          0.0


In [ ]:
## Split dataset randomly
num_model_pairs = 10
data_splits, memberships = split_dataframe_for_training(data_small, num_model_pairs)

## Train cox models and get trained models and metadata
models, metadata = train_cox_models(
    data_splits=data_splits,
    num_model_pairs=num_model_pairs,
    log_dir=global_log_dir
)

## prepare auditing dataset and memberships
auditing_dataset, auditing_membership = sample_cox_auditing_dataset(data_small, memberships)

In [ ]:
## compute attack signals
signals = get_cox_model_signals(models, auditing_dataset, global_log_dir)

In [52]:
# dataset size 400
# Perform the privacy auditing
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.4558, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.5502, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.4914, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.4939, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.5435, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.4800, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.5156, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.5228, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.5265, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.4889, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.5659, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.4752, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.5381, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

In [58]:
# dataset size 1000
# Perform the privacy auditing
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0
Target Model 0: AUC 0.5260, 
Auditing the privacy risks of target model 1
Target Model 1: AUC 0.4720, 
Auditing the privacy risks of target model 2
Target Model 2: AUC 0.4898, 
Auditing the privacy risks of target model 3
Target Model 3: AUC 0.5081, 
Auditing the privacy risks of target model 4
Target Model 4: AUC 0.4927, 
Auditing the privacy risks of target model 5
Target Model 5: AUC 0.5047, 
Auditing the privacy risks of target model 6
Target Model 6: AUC 0.4938, 
Auditing the privacy risks of target model 7
Target Model 7: AUC 0.5209, 
Auditing the privacy risks of target model 8
Target Model 8: AUC 0.4838, 
Auditing the privacy risks of target model 9
Target Model 9: AUC 0.5267, 
Auditing the privacy risks of target model 10
Target Model 10: AUC 0.4857, 
Auditing the privacy risks of target model 11
Target Model 11: AUC 0.5153, 
Auditing the privacy risks of target model 12
Target Model 12: AUC 0.4874, 
Auditing the privacy risks of ta

<Figure size 640x480 with 0 Axes>

## The whole synthetic dataset - size 2000000

In [ ]:
# read the whole synthetic dataset - size 2000000

parquet_directory = "cox_synthetic_dataset/synthetic_dataset"
parquet_files = glob.glob(os.path.join(parquet_directory, "*.parquet"))

dfs = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='pyarrow')  # Specify the PyArrow engine
    dfs.append(df)
data = pd.concat(dfs, ignore_index=True)

#print(data.head())
#print(data.keys())

data.drop('dataset', axis=1, inplace=True) #inplace=True will modify the original df, no need to assign it.

print(f"Total number of rows: {len(data)}")
print(data.head())

## Diversity dataset

In [4]:
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset_diversity/AgeAndSex_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
#data.drop('dataset', axis=1, inplace=True)

print(data.head())
print(len(data))

#global_log_dir = "cox_synthetic_dataset/signal_cindex_dataset_size20000"

         Age       Time  Event  Sex_F  Treatment_B  Treatment_C
0  33.923826   0.244963    1.0    1.0          0.0          1.0
1  36.980792   2.753681    1.0    1.0          0.0          1.0
2  42.434790   2.195357    1.0    1.0          1.0          0.0
3  56.861001  11.350882    1.0    1.0          1.0          0.0
4  76.325600   1.065157    1.0    1.0          0.0          0.0
1000


In [4]:
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset_diversity/DifferentAgePlus10_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
print(data.head())

         Age      Time  Event  Sex_F  Treatment_B  Treatment_C
0  35.358118  2.418039    1.0    1.0          0.0          0.0
1  20.503350  0.123642    1.0    1.0          0.0          1.0
2  75.505561  5.187368    1.0    0.0          1.0          0.0
3  59.145502  3.288964    1.0    1.0          0.0          1.0
4  27.849486  0.255594    1.0    0.0          0.0          1.0


In [5]:
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset_diversity/DifferentAgePlus20_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
print(data.head())
print(len(data))

         Age      Time  Event  Sex_F  Treatment_B  Treatment_C
0  49.562477  2.370392    1.0    1.0          0.0          0.0
1  79.820340  6.301289    1.0    1.0          1.0          0.0
2  36.372892  9.526165    1.0    1.0          0.0          1.0
3  85.052323  1.590701    1.0    0.0          1.0          0.0
4  85.755424  9.159500    1.0    0.0          1.0          0.0
1000


In [6]:
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset_diversity/Reference_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
print(data.head())

         Age       Time  Event  Sex_F  Treatment_B  Treatment_C
0  39.870809   4.643715    1.0    1.0          1.0          0.0
1  49.266457  20.017394    1.0    0.0          1.0          0.0
2  31.305418   3.887630    1.0    1.0          1.0          0.0
3  47.849609   0.615960    1.0    1.0          0.0          1.0
4  42.572500   0.612842    0.0    0.0          0.0          1.0


In [7]:
parquet_file_path = "cox_synthetic_dataset/synthetic_dataset_diversity/Treatment2_iteration1.parquet"
data = pd.read_parquet(parquet_file_path, engine='pyarrow') # Recommended
print(data.head())

         Age      Time  Event  Sex_F  Treatment_B  Treatment_C
0  31.629996  0.579345    1.0    1.0          0.0          1.0
1  54.143722  1.204656    1.0    1.0          1.0          0.0
2  51.452593  6.231999    1.0    0.0          0.0          1.0
3  52.807165  1.108758    1.0    0.0          0.0          1.0
4  44.433126  0.328165    1.0    1.0          0.0          1.0


### AgeAndSex dataset as training

In [22]:
# training dataset - AgeAndSex

parquet_directory = "cox_synthetic_dataset/synthetic_dataset_diversity"
parquet_files = glob.glob(os.path.join(parquet_directory, "AgeAndSex_iteration*.parquet"))

dfs = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='pyarrow')  # Specify the PyArrow engine
    dfs.append(df)
data = pd.concat(dfs, ignore_index=True)

print(f"Total number of rows: {len(data)}")
print(data.head())

Total number of rows: 3000
         Age       Time  Event  Sex_F  Treatment_B  Treatment_C
0  33.923826   0.244963    1.0    1.0          0.0          1.0
1  36.980792   2.753681    1.0    1.0          0.0          1.0
2  42.434790   2.195357    1.0    1.0          1.0          0.0
3  56.861001  11.350882    1.0    1.0          1.0          0.0
4  76.325600   1.065157    1.0    1.0          0.0          0.0


### auditing dataset

In [23]:
# auditing dataset - Plus20

parquet_directory = "cox_synthetic_dataset/synthetic_dataset_diversity"
parquet_files = glob.glob(os.path.join(parquet_directory, "DifferentAgePlus20_iteration*.parquet"))

dfs = []
for file in parquet_files:
    df = pd.read_parquet(file, engine='pyarrow')  # Specify the PyArrow engine
    dfs.append(df)
auditing_data = pd.concat(dfs, ignore_index=True)

print(f"Total number of rows: {len(auditing_data)}")
print(auditing_data.head())

Total number of rows: 3000
         Age      Time  Event  Sex_F  Treatment_B  Treatment_C
0  49.562477  2.370392    1.0    1.0          0.0          0.0
1  79.820340  6.301289    1.0    1.0          1.0          0.0
2  36.372892  9.526165    1.0    1.0          0.0          1.0
3  85.052323  1.590701    1.0    0.0          1.0          0.0
4  85.755424  9.159500    1.0    0.0          1.0          0.0


In [27]:
# if completely use non-overlapping auditing dataset, TPR and FNR mean nothing!!!

def sample_cox_auditing_dataset():
    # auditing membership = 0
    auditing_data_size = int(len(auditing_data) / 2) # 1500
    auditing_membership = np.full((2 * num_model_pairs, len(auditing_data)), True, dtype=bool)
    sample_auditing_data = auditing_data.sample(n=auditing_data_size, random_state=42)
    
    return auditing_data, auditing_membership

In [ ]:
# half training, half non-over

def sample_cox_auditing_dataset():
    # auditing membership = 0
    auditing_data_size = int(len(auditing_data) / 2) # 1500
    auditing_membership = np.full((2 * num_model_pairs, len(auditing_data)), True, dtype=bool)
    sample_auditing_data = auditing_data.sample(n=auditing_data_size, random_state=42)
    
    return auditing_data, auditing_membership

### half of auditing dataset is from Plus20 (heterogeity) dataset, the non-member ones

## workflow

In [24]:
## Split dataset randomly

num_model_pairs = 10
data_splits, memberships = split_dataframe_for_training(data, num_model_pairs)

In [25]:
## Train cox models and get trained models and metadata

models, metadata = train_cox_models(
    data_splits=data_splits,
    num_model_pairs=num_model_pairs,
    log_dir=global_log_dir
)

#print(metadata[0])  # Metadata for the first model
#models[0].print_summary()  # Summary of the first model

In [28]:
## prepare auditing dataset and memberships

auditing_dataset, auditing_membership = sample_cox_auditing_dataset()

In [29]:
## compute attack signals
signals = get_cox_model_signals(models, auditing_dataset, global_log_dir)

Compute attack signals for model 0...
Compute attack signals for model 1...
Compute attack signals for model 2...
Compute attack signals for model 3...
Compute attack signals for model 4...
Compute attack signals for model 5...
Compute attack signals for model 6...
Compute attack signals for model 7...
Compute attack signals for model 8...
Compute attack signals for model 9...
Compute attack signals for model 10...
Compute attack signals for model 11...
Compute attack signals for model 12...
Compute attack signals for model 13...
Compute attack signals for model 14...
Compute attack signals for model 15...
Compute attack signals for model 16...
Compute attack signals for model 17...
Compute attack signals for model 18...
Compute attack signals for model 19...
Signals saved to disk.


### square

In [16]:
global_log_dir = "cox_synthetic_dataset/diversity_train_AgeAndSex_audit_Plus20/signal_square_option2"

In [30]:
num_experiments = num_model_pairs * 2
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_cox_models(
    target_model_indices,
    signals,
    auditing_membership,
    global_log_dir
)

Auditing the privacy risks of target model 0


/home/vesper/.local/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:1179: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


IndexError: index -1 is out of bounds for axis 0 with size 0